<a href="https://colab.research.google.com/github/Asvelasquez/PROCESAMIENTO-DE-LENGUAJE-NATURAL/blob/main/01_texto_a_embeddings_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformacion de texto en representaciones vectoriales con TF IDF

En el siguiente ejercicio se busca transformar textos en español en representaciones numericas mediante TF IDF, calcular similitud del coseno y realizar una búsqueda sencilla de documentos similares.


## 1. librerias

Se utiliza scikit-learn para construir la representación TF IDF y calcular similitud del coseno, numpy para ordenar resultados y librerias estandar de python para el preprocesamiento.

In [12]:
import re
import unicodedata
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. corpus de documentos

Se utiliza un corpus pequeño y controlado de ciberseguridad, cada documento tiene una categoria asociada para facilitar la interpretacion de los resultados.

In [13]:
documentos = ['El atacante envio un correo falso para robar credenciales y contrasenas.', 'La campana de phishing busca obtener datos bancarios mediante un enlace malicioso.', 'Los empleados recibieron un mensaje falso que imitaba al banco.', 'El portal fraudulento solicita usuario, contrasena y codigo de verificacion.', 'El malware roba credenciales almacenadas en el navegador.', 'Un troyano bancario fue detectado en varios equipos de la empresa.', 'El programa malicioso establece una conexion con el atacante.', 'El equipo de seguridad bloqueo el malware y analizo la evidencia.', 'El ransomware cifra los archivos del servidor y exige un rescate.', 'Las copias de seguridad permitieron recuperar los archivos sin pagar.', 'El atacante ingreso al servidor mediante una vulnerabilidad.', 'La actualizacion de seguridad corrigio la vulnerabilidad critica.', 'El exploit permite ejecutar codigo en el servidor vulnerable.', 'El analista recomendo instalar el parche de seguridad.', 'La fuga de datos expuso informacion de clientes.', 'El monitoreo detecto una cuenta falsa que imitaba a la empresa.', 'El equipo solicito el cierre del sitio fraudulento.', 'La empresa reforzo la seguridad despues del incidente.', 'El servidor fue revisado para identificar riesgos de seguridad.', 'El informe describe un ataque de phishing contra empleados.']
categorias = ['phishing', 'phishing', 'phishing', 'phishing', 'malware', 'malware', 'malware', 'malware', 'ransomware', 'ransomware', 'vulnerabilidad', 'vulnerabilidad', 'vulnerabilidad', 'vulnerabilidad', 'fuga', 'fuga', 'fuga', 'general', 'general', 'phishing']

print(f"Cantidad de documentos: {len(documentos)}")
print(f"Cantidad de categorías: {len(set(categorias))}")

Cantidad de documentos: 20
Cantidad de categorías: 6


## 3. Preprocesamiento

Antes de convertir los textos en vectores se realizan las siguientes operaciones:

1. Convertir el texto a minusculas.
2. Normalizar y retirar tildes para reducir variaciones superficiales.
3. Extraer palabras mediante expresiones regulares.
4. Eliminar palabras muy cortas y algunas palabras funcionales frecuentes

In [14]:
STOPWORDS = {
    "el", "la", "los", "las", "un", "una", "unos", "unas",
    "de", "del", "y", "o", "en", "por", "para", "con",
    "sin", "al", "que", "se", "su", "sus", "a"
}

def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto if not unicodedata.combining(c))

def tokenizar(texto):
    texto = normalizar(texto)
    tokens = re.findall(r"[a-zñ]+", texto)
    return [t for t in tokens if len(t) >= 3 and t not in STOPWORDS]

documentos_procesados = [" ".join(tokenizar(doc)) for doc in documentos]

for i in range(3):
    print(f"Documento {i+1}: {documentos_procesados[i]}")

Documento 1: atacante envio correo falso robar credenciales contrasenas
Documento 2: campana phishing busca obtener datos bancarios mediante enlace malicioso
Documento 3: empleados recibieron mensaje falso imitaba banco


## 4.TF IDF

TF IDF (Term Frequency - Inverse Document Frequency) representa cada documento como un vector numerico, el peso de un termino aumenta cuando aparece con frecuencia en un documento, pero disminuye si aparece en muchos documentos del corpus, en este ejercicio se utiliza TfidfVectorizer con normalizacion L2.

In [15]:
def crear_representacion_tfidf(documentos):
    vectorizador = TfidfVectorizer(lowercase=False, norm="l2")
    X = vectorizador.fit_transform(documentos)
    vocabulario = vectorizador.get_feature_names_out()
    return vectorizador, X, vocabulario

vectorizador, X, vocabulario = crear_representacion_tfidf(documentos_procesados)

print(f"documentos: {X.shape[0]}")
print(f"terminos del vocabulario: {X.shape[1]}")
print(f"dimensiones de la matriz TF IDF: {X.shape}")

documentos: 20
terminos del vocabulario: 91
dimensiones de la matriz TF IDF: (20, 91)


## 5. Revision de la representacion vectorial

Cada fila de X corresponde a un documento y cada columna corresponde a un termino del vocabulario, el valor almacenado representa el peso TF IDF de ese termino en el documento.

In [16]:
print("primeros 20 terminos del vocabulario:")
print(vocabulario[:20])

fila = X[0].toarray()[0]
indices = np.argsort(fila)[::-1]

print("\nterminos con mayor peso en el documento 1:")
mostrados = 0
for idx in indices:
    if fila[idx] > 0:
        print(f"{vocabulario[idx]:20s} -> {fila[idx]:.4f}")
        mostrados += 1
        if mostrados == 8:
            break

primeros 20 terminos del vocabulario:
['actualizacion' 'almacenadas' 'analista' 'analizo' 'archivos' 'atacante'
 'ataque' 'bancario' 'bancarios' 'banco' 'bloqueo' 'busca' 'campana'
 'cierre' 'cifra' 'clientes' 'codigo' 'conexion' 'contra' 'contrasena']

terminos con mayor peso en el documento 1:
robar                -> 0.4024
contrasenas          -> 0.4024
envio                -> 0.4024
correo               -> 0.4024
falso                -> 0.3538
credenciales         -> 0.3538
atacante             -> 0.3192


## 6. Similitud de coseno

Para comparar documentos se utiliza la similitud del coseno, esta medida compara el angulo entre sus vectores TF IDF.

- Un valor cercano a 1 indica alta similitud.
- Un valor cercano a 0 indica poca similitud.

se comparan los dos primeros documentos, ambos relacionados con phishing

In [17]:
def calcular_similitud(vector_a, vector_b):
    return cosine_similarity(vector_a, vector_b)[0][0]

doc_a, doc_b = 0, 1
similitud = calcular_similitud(X[doc_a], X[doc_b])

print("documento 1:", documentos[doc_a])
print("documento 2:", documentos[doc_b])
print(f"\nsimilitud del coseno: {similitud:.4f}")

documento 1: El atacante envio un correo falso para robar credenciales y contrasenas.
documento 2: La campana de phishing busca obtener datos bancarios mediante un enlace malicioso.

similitud del coseno: 0.0000


## 7. busqueda de documentos similares

A traves de TF IDF se busca convertir una consulta del usuario en el mismo espacio vectorial y ordenar los documentos segun su similitud con la consulta.

In [18]:
def buscar(consulta, top_k=5):
    consulta_procesada = " ".join(tokenizar(consulta))
    vector_consulta = vectorizador.transform([consulta_procesada])
    similitudes = cosine_similarity(vector_consulta, X)[0]
    indices = np.argsort(similitudes)[::-1][:top_k]
    return [(i, similitudes[i], categorias[i], documentos[i]) for i in indices]

consulta = "correo falso para robar contrasenas"
resultados = buscar(consulta)

print("consulta:", consulta)
print("\nresultados:")
for i, score, categoria, texto in resultados:
    print(f"{score:.4f} | {categoria:16s} | documento {i+1}: {texto}")

consulta: correo falso para robar contrasenas

resultados:
0.7817 | phishing         | documento 1: El atacante envio un correo falso para robar credenciales y contrasenas.
0.1725 | phishing         | documento 3: Los empleados recibieron un mensaje falso que imitaba al banco.
0.0000 | phishing         | documento 20: El informe describe un ataque de phishing contra empleados.
0.0000 | general          | documento 19: El servidor fue revisado para identificar riesgos de seguridad.
0.0000 | fuga             | documento 17: El equipo solicito el cierre del sitio fraudulento.


## 8. prueba con diferentes consultas

Para analizar el comportamiento de TF IDF se utilizan varias consultas relacionadas con diferentes temas del corpus, cada consulta se transforma al mismo espacio vectorial y se obtienen los documentos con mayor similitud.

In [19]:
consultas = [
    "correo falso para robar contrasenas",
    "malware en equipos de la empresa",
    "archivos cifrados por ransomware",
    "vulnerabilidad en un servidor",
    "fuga de informacion de clientes"
]

for consulta in consultas:
    resultados = buscar(consulta, top_k=3)

    print("\nconsulta:", consulta)
    print("resultados:")

    for i, score, categoria, texto in resultados:
        print(f"{score:.4f} | {categoria:16s} | documento {i+1}: {texto}")


consulta: correo falso para robar contrasenas
resultados:
0.7817 | phishing         | documento 1: El atacante envio un correo falso para robar credenciales y contrasenas.
0.1725 | phishing         | documento 3: Los empleados recibieron un mensaje falso que imitaba al banco.
0.0000 | phishing         | documento 20: El informe describe un ataque de phishing contra empleados.

consulta: malware en equipos de la empresa
resultados:
0.4155 | malware          | documento 6: Un troyano bancario fue detectado en varios equipos de la empresa.
0.2339 | malware          | documento 5: El malware roba credenciales almacenadas en el navegador.
0.2244 | malware          | documento 8: El equipo de seguridad bloqueo el malware y analizo la evidencia.

consulta: archivos cifrados por ransomware
resultados:
0.5783 | ransomware       | documento 9: El ransomware cifra los archivos del servidor y exige un rescate.
0.2554 | ransomware       | documento 10: Las copias de seguridad permitieron recuperar

## 9. resultados

la representacion TF IDF permite identificar documentos relacionados con una consulta cuando comparten terminos relevantes, por ejemplo palabras como correo, falso,robar y contrasenas tienen importancia porque ayudan a distinguir el contenido de la consulta.

el metodo es sencillo, rapido e interpretable, sin embargo, tiene una limitacion importante, no representa directamente el significado semantico de las palabras, por ejemplo, dos palabras sin coincidencia pueden ser similares y TF IDF puede asignarles una similitud baja o nula.

esto permite diferenciar TF IDF de modelos distribucionales como word2vec, incluidos en el laboratorio original, wl laboratorio original muestra, por ejemplo, que TF IDF puede tener similitud 0 entre terminos que no comparten vocabulario, mientras word2vec puede capturar relaciones semanticas.

## 10. conclusiones

el texto puede transformarse en una representacion numerica mediante TF IDF, mientras que el preprocesamiento permite reducir variaciones innecesarias y mejorar la representacion de los documentos, a partir de los vectores generados, la similitud del coseno permite comparar documentos y consultas dentro de un espacio vectorial,  TF IDF es adecuado en este caso debido a que es una tecnica sencilla de interpretar e implementar, pero su principal limitacion es que depende de la coincidencia de terminos y no captura por si solo relaciones semanticas profundas entre las palabras, por esas razones, TF-IDF constituye una alternativa sencilla para cumplir el objetivo de transformar texto en representaciones vectoriales.
